# Deteksi Penipuan Wajah (Face Spoofing) Menggunakan Ensemble Learning

## Import Library

Notebook ini mengimplementasikan solusi untuk deteksi penipuan wajah (face spoofing) menggunakan ensemble dari dua model deep learning yang sudah terlatih: EfficientNet-B0 dan ResNet18. Tujuannya adalah untuk mengklasifikasikan gambar ke dalam beberapa kategori, membedakan antara 'realperson' dan berbagai jenis wajah 'palsu' (misalnya, topeng, manekin, cetakan, layar, tidak dikenal).

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
import torch.nn.functional as F
from collections import Counter
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

Bagian ini mengimpor semua pustaka yang diperlukan untuk manipulasi data, pembangunan model deep learning, pelatihan, dan evaluasi. Pustaka utama meliputi `torch` untuk jaringan saraf, `torchvision` untuk tugas-tugas visi komputer dan model-model yang sudah terlatih, `PIL` untuk pemrosesan gambar, `pandas` untuk penanganan data, dan `sklearn` untuk metrik evaluasi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Untuk mengakses dataset yang disimpan di Google Drive, notebook ini melakukan mount Drive ke lingkungan Colab. Ini memungkinkan kode untuk membaca dan menulis file dari jalur yang ditentukan.

Load Data

## Konfigurasi dan Jalur Data

Bagian ini mendefinisikan konstanta global dan jalur yang digunakan di seluruh notebook, termasuk direktori untuk data pelatihan dan pengujian, ukuran gambar, ukuran batch, jumlah epoch, laju pembelajaran, dan konfigurasi perangkat (GPU atau CPU). Ini juga mendefinisikan ekstensi file gambar yang valid.

In [ ]:
TRAIN_DIR = "/content/drive/MyDrive/Submission Kaggle/train baru"
TEST_DIR = "/content/drive/MyDrive/Submission Kaggle/test"
SAMPLE_SUB_PATH = "/content/drive/MyDrive/Submission Kaggle/samplesubmission.csv"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

In [ ]:
label_mapping = {
    'realperson':0,
    'fake_mask':1,
    'fake_mannequin':2,
    'fake_printed':3,
    'fake_screen':4,
    'fake_unknown':5
}

inv_label_mapping = {v:k for k,v in label_mapping.items()}

## Pemetaan Label

Sebuah kamus `label_mapping` dibuat untuk mengonversi label string kategorikal (misalnya, 'realperson', 'fake_mask') menjadi label bilangan bulat numerik, yang diperlukan untuk pelatihan model. Pemetaan terbalik `inv_label_mapping` juga dibuat untuk mengonversi prediksi numerik kembali ke label yang mudah dibaca manusia.

Cleaning Data

## Pembersihan Data

Fungsi `find_invalid_images` mengidentifikasi dan menghapus file yang rusak atau bukan file gambar dari direktori dataset. Langkah ini sangat penting untuk memastikan bahwa pemuat data hanya memproses file gambar yang valid, mencegah potensi kesalahan selama pelatihan dan inferensi.

In [ ]:
# Pengecekan file corrupt atau bukan image
def find_invalid_images(folder_path):
    bad_files = []
    non_image_files = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            if not file.lower().endswith(VALID_EXTENSIONS):
                non_image_files.append(file_path)
                continue
            try:
                img = Image.open(file_path)
                img.verify()
            except Exception:
                bad_files.append(file_path)
    return non_image_files, bad_files

non_image_train, bad_train = find_invalid_images(TRAIN_DIR)
for f in non_image_train + bad_train:
    os.remove(f)
print(f"Data cleaning selesai. Dihapus {len(non_image_train)+len(bad_train)} file bermasalah.")


Data cleaning selesai. Dihapus 0 file bermasalah.


Define Custom Dataset

## Kelas Dataset Kustom

Dua kelas `Dataset` PyTorch kustom didefinisikan untuk menangani pemuatan gambar pelatihan dan pengujian:

-   **`FaceSpoofTrainDataset`**: Mengulangi direktori pelatihan, mengumpulkan jalur gambar dan label yang sesuai (dipetakan ke bilangan bulat), dan menerapkan transformasi yang ditentukan. Ini mengharapkan subfolder untuk merepresentasikan label kelas.
-   **`FaceSpoofTestDataset`**: Mengulangi direktori pengujian, mengumpulkan jalur gambar, dan menerapkan transformasi. Ini tidak menetapkan label karena ini tidak diketahui untuk set pengujian.

In [ ]:
class FaceSpoofTrainDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for subfolder in os.listdir(root):
            label = label_mapping[subfolder]
            folder_path = os.path.join(root, subfolder)
            for f in os.listdir(folder_path):
                if f.lower().endswith(VALID_EXTENSIONS):
                    self.image_paths.append(os.path.join(folder_path, f))
                    self.labels.append(label)
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label

In [ ]:
class FaceSpoofTestDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.image_paths = sorted([os.path.join(root, f) for f in os.listdir(root) if f.lower().endswith(VALID_EXTENSIONS)])
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(path)

## Augmentasi dan Pra-pemrosesan Data

Bagian ini mendefinisikan transformasi gambar yang diterapkan pada dataset pelatihan dan validasi/pengujian:

-   **`train_transforms`**: Meliputi teknik augmentasi data seperti `RandomResizedCrop`, `RandomHorizontalFlip`, `RandomRotation`, `ColorJitter`, dan `RandomGrayscale` untuk meningkatkan generalisasi model. Semua gambar dikonversi menjadi tensor dan dinormalisasi menggunakan mean dan standar deviasi ImageNet.
-   **`val_transforms`**: Terdiri dari pengubahan ukuran, konversi ke tensor, dan normalisasi, tanpa augmentasi, untuk evaluasi yang konsisten.

## Pengolahan / Augmentasi Data
1.  Augmentasi yang digunakan:
    *   `RandomResizedCrop`: memotong dan mengubah ukuran gambar secara acak,
    *   `RandomHorizontalFlip`: membalik gambar secara horizontal,
    *   `RandomRotation`: rotasi kecil (<15°),
    *   `ColorJitter`: mengubah kecerahan/kontras ringan,
    *   `RandomGrayscale`: mengubah beberapa gambar menjadi skala abu-abu.
2.  Normalisasi menggunakan mean/std ImageNet.

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

## Inisialisasi Dataset dan DataLoader

Di sini, instance `FaceSpoofTrainDataset` dan `FaceSpoofTestDataset` dibuat, menerapkan transformasi yang telah ditentukan. Dataset ini kemudian dibungkus dalam objek `DataLoader`, yang menangani batching, pengacakan (untuk pelatihan), dan memuat data secara efisien ke model.

Load Dataset

In [ ]:
train_dataset = FaceSpoofTrainDataset(TRAIN_DIR, transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataset = FaceSpoofTestDataset(TEST_DIR, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Perhitungan Bobot Kelas

Untuk mengatasi potensi ketidakseimbangan kelas dalam dataset pelatihan, bobot kelas dihitung. Modul `Counter` digunakan untuk menghitung kejadian setiap label, dan bobot dihitung sedemikian rupa sehingga kelas yang kurang sering muncul menerima bobot yang lebih tinggi. Bobot ini kemudian diteruskan ke fungsi `CrossEntropyLoss` selama pelatihan.

In [ ]:
labels_count = Counter(train_dataset.labels)
total_count = sum(labels_count.values())
class_weights = [total_count/labels_count[i] for i in range(len(label_mapping))]
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

## Arsitektur dan Inisialisasi Model

Bagian ini menyiapkan dua model deep learning untuk pembelajaran ensemble:

1.  **EfficientNet-B0**: Model EfficientNet-B0 yang sudah terlatih dimuat. Lapisan klasifikasi akhir diganti agar sesuai dengan jumlah kelas keluaran di dataset kami.
2.  **ResNet18**: Demikian pula, model ResNet18 yang sudah terlatih dimuat, dan lapisan terhubung penuh terakhirnya dimodifikasi untuk tugas klasifikasi spesifik kami.

Kedua model dipindahkan ke `DEVICE` yang ditentukan (GPU jika tersedia, jika tidak, CPU). `criterion` (CrossEntropyLoss dengan bobot kelas), `optimizers` (Adam), dan `learning rate schedulers` (CosineAnnealingLR) didefinisikan untuk setiap model secara independen.

## Pemodelan
1.  Model yang sudah dilatih digunakan (EfficientNet-B0 dan ResNet18).
2.  Fine-tuning seluruh lapisan untuk memanfaatkan bobot yang sudah dilatih.
3.  Weighted loss digunakan untuk mengatasi ketidakseimbangan antar kelas.

In [ ]:
# EfficientNet-B0
model1 = models.efficientnet_b0(pretrained=True)
num_features = model1.classifier[1].in_features
model1.classifier[1] = nn.Linear(num_features, len(label_mapping))
model1 = model1.to(DEVICE)

# ResNet18
model2 = models.resnet18(pretrained=True)
num_features2 = model2.fc.in_features
model2.fc = nn.Linear(num_features2, len(label_mapping))
model2 = model2.to(DEVICE)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most u

## Loop Pelatihan

Loop pelatihan berulang untuk jumlah `EPOCHS` yang ditentukan. Di setiap epoch, kedua EfficientNet-B0 (`model1`) dan ResNet18 (`model2`) dilatih secara terpisah:

-   Model diatur ke mode pelatihan (`model.train()`).
-   Untuk setiap batch gambar, prediksi dibuat, loss dihitung menggunakan `CrossEntropyLoss` berbobot, dan backpropagation dilakukan.
-   Loss pelatihan, akurasi, dan F1-score makro dilaporkan untuk setiap epoch.
-   Scheduler laju pembelajaran di-step setelah setiap epoch untuk menyesuaikan laju pembelajaran.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer1 = optim.Adam(model1.parameters(), lr=LR)
optimizer2 = optim.Adam(model2.parameters(), lr=LR)

scheduler1 = optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=EPOCHS)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=EPOCHS)

## Loop Pelatihan

In [ ]:
for epoch in range(EPOCHS):
    # --- EfficientNet ---
    model1.train()
    running_loss1, correct1, total1 = 0,0,0
    all_labels, all_preds = [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer1.zero_grad()
        outputs = model1(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer1.step()
        running_loss1 += loss.item()
        _, preds = torch.max(outputs, 1)
        correct1 += (preds == labels).sum().item()
        total1 += labels.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"Epoch {epoch+1} EfficientNet - Loss: {running_loss1/len(train_loader):.4f}, Acc: {correct1/total1:.4f}, Macro-F1: {macro_f1:.4f}")
    scheduler1.step()

    # --- ResNet18 ---
    model2.train()
    running_loss2, correct2, total2 = 0,0,0
    all_labels2, all_preds2 = [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer2.zero_grad()
        outputs = model2(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer2.step()
        running_loss2 += loss.item()
        _, preds = torch.max(outputs, 1)
        correct2 += (preds == labels).sum().item()
        total2 += labels.size(0)
        all_labels2.extend(labels.cpu().numpy())
        all_preds2.extend(preds.cpu().numpy())
    macro_f12 = f1_score(all_labels2, all_preds2, average='macro')
    print(f"Epoch {epoch+1} ResNet18 - Loss: {running_loss2/len(train_loader):.4f}, Acc: {correct2/total2:.4f}, Macro-F1: {macro_f12:.4f}")
    scheduler2.step()

Epoch 1 EfficientNet - Loss: 1.3830, Acc: 0.5788, Macro-F1: 0.5678
Epoch 1 ResNet18 - Loss: 0.7211, Acc: 0.7523, Macro-F1: 0.7425
Epoch 2 EfficientNet - Loss: 0.6542, Acc: 0.8405, Macro-F1: 0.8414
Epoch 2 ResNet18 - Loss: 0.2396, Acc: 0.9264, Macro-F1: 0.9268
Epoch 3 EfficientNet - Loss: 0.3673, Acc: 0.8984, Macro-F1: 0.8977
Epoch 3 ResNet18 - Loss: 0.1301, Acc: 0.9647, Macro-F1: 0.9654
Epoch 4 EfficientNet - Loss: 0.2464, Acc: 0.9294, Macro-F1: 0.9296
Epoch 4 ResNet18 - Loss: 0.0879, Acc: 0.9757, Macro-F1: 0.9756
Epoch 5 EfficientNet - Loss: 0.1740, Acc: 0.9495, Macro-F1: 0.9513
Epoch 5 ResNet18 - Loss: 0.0735, Acc: 0.9805, Macro-F1: 0.9808
Epoch 6 EfficientNet - Loss: 0.1529, Acc: 0.9598, Macro-F1: 0.9603
Epoch 6 ResNet18 - Loss: 0.0442, Acc: 0.9884, Macro-F1: 0.9889
Epoch 7 EfficientNet - Loss: 0.1330, Acc: 0.9617, Macro-F1: 0.9610
Epoch 7 ResNet18 - Loss: 0.0463, Acc: 0.9915, Macro-F1: 0.9917
Epoch 8 EfficientNet - Loss: 0.1116, Acc: 0.9738, Macro-F1: 0.9744
Epoch 8 ResNet18 - Loss

## Evaluasi dan Prediksi Ensemble

Setelah pelatihan, kedua model diatur ke mode evaluasi (`model.eval()`). Prediksi dihasilkan untuk dataset pengujian menggunakan pendekatan ensemble:

-   Probabilitas softmax dari kedua model (`out1` dan `out2`) dirata-ratakan untuk mendapatkan `out_ensemble`.
-   Kelas dengan probabilitas rata-rata tertinggi dipilih sebagai prediksi akhir.
-   Prediksi kemudian dipetakan kembali ke label string aslinya menggunakan `inv_label_mapping`.

## Evaluasi

In [ ]:
model1.eval(); model2.eval()
ensemble_preds, image_ids = [], []

with torch.no_grad():
    for imgs, paths in test_loader:
        imgs = imgs.to(DEVICE)
        out1 = F.softmax(model1(imgs), dim=1)
        out2 = F.softmax(model2(imgs), dim=1)
        out_ensemble = (out1 + out2)/2
        preds = out_ensemble.argmax(dim=1)
        ensemble_preds.extend(preds.cpu().numpy())
        image_ids.extend(paths)

ensemble_labels_mapped = [inv_label_mapping[p] for p in ensemble_preds]


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


## Pembuatan File Pengajuan (Submission)

Bagian ini membuat file pengajuan akhir dalam format yang disyaratkan:

-   Sebuah DataFrame pandas dibangun dengan ID gambar dan prediksi ensemble yang sesuai.
-   DataFrame ini digabungkan dengan file pengajuan sampel untuk memastikan semua gambar pengujian disertakan.
-   Setiap label yang hilang (jika ada) diisi dengan 'fake_unknown'.
-   Pengajuan akhir disimpan sebagai file CSV bernama `submission_face_spoofing_macroF1_ensemble.csv`.

In [ ]:
submission_df = pd.DataFrame({
    'id':[os.path.splitext(p)[0] for p in image_ids],
    'label':ensemble_labels_mapped
})

sample_df = pd.read_csv(SAMPLE_SUB_PATH)
final_submission = sample_df[['id']].merge(submission_df, on='id', how='left')
final_submission['label'].fillna('fake_unknown', inplace=True)
final_submission.to_csv("submission_face_spoofing_macroF1_ensemble.csv", index=False)

/tmp/ipykernel_36230/2525824247.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_submission['label'].fillna('fake_unknown', inplace=True)


## Penyimpanan Model

Akhirnya, kamus status terlatih dari kedua `model1` (EfficientNet-B0) dan `model2` (ResNet18) disimpan ke disk sebagai file `.pth`. Ini memungkinkan model terlatih untuk dimuat dan digunakan kembali nanti tanpa pelatihan ulang.

## Menyimpan Model

## Kesimpulan

Notebook ini menyediakan solusi komprehensif untuk deteksi penipuan wajah, memanfaatkan model yang sudah dilatih, augmentasi data, pembobotan kelas untuk penanganan ketidakseimbangan, dan pendekatan ensemble untuk prediksi yang kuat. Proses pelatihan dan langkah-langkah evaluasi didefinisikan dengan jelas, yang berpuncak pada file pengajuan dan model yang disimpan untuk penggunaan di masa mendatang.

In [ ]:
torch.save(model1.state_dict(), "face_spoofing_model1.pth")
torch.save(model2.state_dict(), "face_spoofing_model2.pth")

print("Submission and models saved successfully.")

Submission and models saved successfully.
